# D20 S2 A2 - Saliency and Occlusion Sensitivity
Implements pixel-gradient saliency and black-box occlusion sensitivity using NumPy.


In [1]:
import numpy as np

grad = np.array([[[ 1., -9.], [0.,  2.]],
                 [[-4.,  3.], [7., -1.]]])

img = np.zeros((6, 6))
img[2:4, 2:4] = 10.0

def score_fn(x):
    return float(np.asarray(x)[2:4, 2:4].sum())


In [2]:
def saliency_map(grad):
    grad = np.asarray(grad)
    return np.max(np.abs(grad), axis=0)


def normalize01(x):
    x = np.asarray(x, dtype=float)
    x_min = x.min()
    x_max = x.max()
    if x_max == x_min:
        return np.zeros_like(x, dtype=float)
    return (x - x_min) / (x_max - x_min)


def occlude(image, top, left, size, fill=0.0):
    result = np.array(image, copy=True)
    result[top:top + size, left:left + size] = fill
    return result


def occlusion_map(image, score_fn, size, stride, fill=0.0):
    image = np.asarray(image)
    height, width = image.shape
    baseline_score = score_fn(image)

    tops = range(0, height - size + 1, stride)
    lefts = range(0, width - size + 1, stride)
    sensitivity = np.empty((len(tops), len(lefts)), dtype=float)

    for row, top in enumerate(tops):
        for col, left in enumerate(lefts):
            occluded_image = occlude(image, top, left, size, fill)
            sensitivity[row, col] = baseline_score - score_fn(occluded_image)

    return sensitivity


In [3]:
print(saliency_map(grad))
sens = occlusion_map(img, score_fn, size=2, stride=2, fill=0.0)
print(sens.shape)
print(sens)


[[4. 9.]
 [7. 2.]]
(3, 3)
[[ 0.  0.  0.]
 [ 0. 40.  0.]
 [ 0.  0.  0.]]


In [4]:
# Checks corresponding to all 8 autograder requirements
sal = saliency_map(grad)
assert np.array_equal(sal, np.array([[4., 9.], [7., 2.]]))
assert sal.shape == (2, 2)
assert np.all(sal >= 0)

norm_sal = normalize01(sal)
assert np.isclose(norm_sal.min(), 0.0)
assert np.isclose(norm_sal.max(), 1.0)
assert np.array_equal(normalize01(np.full((2, 2), 3.0)), np.zeros((2, 2)))

original = np.arange(36, dtype=float).reshape(6, 6)
before = original.copy()
occluded = occlude(original, 2, 1, 2)
assert np.array_equal(original, before)
assert np.all(occluded[2:4, 1:3] == 0.0)
assert np.array_equal(occluded[:2], original[:2])

sens = occlusion_map(img, score_fn, size=2, stride=2, fill=0.0)
assert sens.shape == (3, 3)
assert np.unravel_index(np.argmax(sens), sens.shape) == (1, 1)
expected = np.array([[0., 0., 0.],
                     [0., 40., 0.],
                     [0., 0., 0.]])
assert np.array_equal(sens, expected)

filled = occlude(np.zeros((4, 4)), 1, 1, 2, fill=5.0)
assert np.all(filled[1:3, 1:3] == 5.0)

print("All 8 autograder-style checks passed.")


All 8 autograder-style checks passed.
